# MoveScope 消融实验

对比二维、三维、加权 DTW 和完整加权分段 DTW 在正确与错误深蹲视频上的表现。

In [ ]:
from pathlib import Path
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "movescope").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

from movescope.experiments import run_ablation, summarize_ablation, run_viewpoint_consistency, viewpoint_std
from movescope.template import ActionTemplate

FIG_DIR = ROOT / "docs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
template_path = Path("data/templates/squat.npz")
if not template_path.exists():
    print("缺少 data/templates/squat.npz，请先运行 scripts/build_template.py 构建模板。")
    rows = []
else:
    template = ActionTemplate.load("squat")
    rows = run_ablation(template, good_dir="data/test/good_squat", bad_dir="data/test/bad_squat")
    print(f"已收集 {len(rows)} 条评分记录")


In [ ]:
df = pd.DataFrame(rows)
if df.empty:
    print("未找到可评分视频，请先向 data/test/good_squat 和 data/test/bad_squat 添加数据。")
else:
    display(df.head())
    display(pd.DataFrame(summarize_ablation(rows)))


In [ ]:
if not df.empty:
    fig, ax = plt.subplots(figsize=(9, 5))
    df.boxplot(column="total_score", by=["variant", "label"], ax=ax, rot=30)
    ax.set_title("不同方法在正确与错误动作上的评分分布")
    ax.set_xlabel("方法与动作标签")
    ax.set_ylabel("总分")
    fig.suptitle("")
    fig.tight_layout()
    output = FIG_DIR / "ablation_result.png"
    fig.savefig(output, dpi=160)
    print(f"图表已保存：{output}")


In [ ]:
if not df.empty:
    for variant in sorted(df["variant"].unique()):
        good = df[(df.variant == variant) & (df.label == "good")]["total_score"]
        bad = df[(df.variant == variant) & (df.label == "bad")]["total_score"]
        if len(good) >= 2 and len(bad) >= 2:
            test = stats.ttest_ind(good, bad, equal_var=False)
            print(f"{variant}：评分区分度={good.mean() - bad.mean():.2f}，p={test.pvalue:.4g}")
        else:
            print(f"{variant}：t 检验至少需要 2 个正确动作视频和 2 个错误动作视频")
